# Download Arabidopsis WGS reads (first N pairs) on Colab, save to Google Drive
Colab downloads from ENA at high speed. This notebook streams each run, keeps only the
first `n_pairs` read pairs (~15x coverage), checks the line counts, and saves to Google Drive.
Afterwards, download the Drive folder `genom_project/raw_sra` to your laptop into
`d:\genom_project\data\raw_sra\`.

**Drive space:** about 13 GB in total. Free Google Drive is 15 GB, so make sure it is mostly empty,
or put only a few accessions in `SAMPLES` at a time.

Run the cells top to bottom (Runtime > Run all).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get -qq install -y pigz > /dev/null && echo pigz ready
!df -h /content | tail -1

In [ ]:
# sample, run, n_pairs  (copied from config/samples.tsv)
SAMPLES = [
    ("Col-0",           "SRR1945757",  9000000),
    ("An-1",            "SRR1945750",  8738816),
    ("Cvi-0",           "SRR1945758",  9000000),
    ("Ler-1",           "SRR1945772",  9000000),
    ("Kyoto",           "SRR1945866",  9000000),
    ("Ale-Stenar-44-4", "SRR1945477", 11800000),
    ("Se-0",            "SRR1945785",  9000000),
    ("Kondara",         "SRR1945770",  9000000),
]
LOCAL = "/content/raw"                                   # fast Colab disk
DRIVE = "/content/drive/MyDrive/genom_project/raw_sra"   # final copy on Drive

import os
os.makedirs(LOCAL, exist_ok=True)
os.makedirs(DRIVE, exist_ok=True)

def url(run, r):
    # ENA layout: vol1/fastq/<first 6 chars>/00<last digit>/<run>/<run>_<r>.fastq.gz
    return f"https://ftp.sra.ebi.ac.uk/vol1/fastq/{run[:6]}/00{run[-1]}/{run}/{run}_{r}.fastq.gz"

for s, run, n in SAMPLES:
    print(f"{s:16s} {url(run, 1)}")

In [ ]:
import subprocess, time
from concurrent.futures import ThreadPoolExecutor

def fetch(sample, run, n, r):
    out = f"{LOCAL}/{sample}_R{r}.fastq.gz"
    final = f"{DRIVE}/{sample}_R{r}.fastq.gz"
    if os.path.exists(final + ".ok"):
        return f"skip {sample} R{r} (already on Drive)"
    for attempt in range(1, 6):
        t0 = time.time()
        # stream -> keep the first n reads -> recompress.
        # head closes the pipe early on purpose, so curl/gzip 'broken pipe' errors are expected.
        cmd = (f"curl -sS --retry 5 '{url(run, r)}' 2>/dev/null | gzip -dc 2>/dev/null "
               f"| head -n {n * 4} | pigz -p 2 > '{out}'")
        subprocess.run(cmd, shell=True, executable="/bin/bash")
        res = subprocess.run(f"pigz -dc '{out}' | wc -l", shell=True, capture_output=True, text=True)
        lines = int(res.stdout.strip() or 0)
        if lines == n * 4:
            subprocess.run(["cp", out, final], check=True)
            with open(final + ".ok", "w") as fh:
                fh.write(f"{run} {n}\n")
            os.remove(out)
            return f"OK   {sample} R{r}  {n:,} reads  {time.time() - t0:.0f}s"
        print(f"retry {attempt} {sample} R{r}: got {lines} lines, want {n * 4}", flush=True)
    return f"FAIL {sample} R{r}"

jobs = [(s, run, n, r) for s, run, n in SAMPLES for r in (1, 2)]
with ThreadPoolExecutor(4) as ex:
    for msg in ex.map(lambda j: fetch(*j), jobs):
        print(msg, flush=True)

In [ ]:
# Final check: every file should have a matching .ok marker
!ls -lh /content/drive/MyDrive/genom_project/raw_sra/
!du -sh /content/drive/MyDrive/genom_project/raw_sra/